# Modeling: Artist Genre Clustering & Feature Analysis

This notebook clusters the 300 simulated users produced by `preprocessing.ipynb`.
Each user is represented by scaled audio features and one-hot genre preferences
across music, podcasts, and books.

1. **Load Data** — read `data/user_features.csv` from preprocessing
2. **Feature Overview** — inspect the feature matrix and genre prevalence
3. **Hyperparameter Tuning** — find optimal k via silhouette score + elbow curve
4. **K-Means Clustering** — cluster users and visualise in 2-D PCA space
5. **Feature Importance** — identify which features most distinguish clusters

## 0. Setup

In [ ]:
import polars as pl
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, cross_val_score
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    classification_report,
    balanced_accuracy_score,
)
from sklearn.decomposition import PCA

print('Libraries loaded.')

## 1. Load Data

In [ ]:
spotify_df = pl.read_csv('data/user_features.csv')

print(f'Spotify tracks: {spotify_df.height}')
print(f'Columns: {spotify_df.columns}')
spotify_df.head(3)

## 2. Feature Overview

The preprocessing pipeline produced three groups of features for each user:
- **Audio features** (6, already min-max scaled): `avg_popularity`, `avg_danceability`,
  `avg_energy`, `avg_valence`, `avg_acousticness`, `avg_tempo`
- **Genre one-hots** (3 × n_genres): `music_*`, `podcast_*`, `book_*`

We build the full feature matrix from these columns, leaving out `user_id` and
the `n_*_genres` count columns.

In [ ]:
AUDIO_COLS = ['avg_popularity', 'avg_danceability', 'avg_energy',
              'avg_valence', 'avg_acousticness', 'avg_tempo']

# All one-hot genre columns
genre_cols = [c for c in spotify_df.columns if c.startswith(('music_', 'podcast_', 'book_'))]

feature_cols = AUDIO_COLS + genre_cols
feature_matrix = spotify_df.select(feature_cols).to_numpy()

print(f'Audio features:  {len(AUDIO_COLS)}')
print(f'Genre one-hots:  {len(genre_cols)}')
print(f'Total features:  {feature_matrix.shape[1]}')
print(f'Feature matrix:  {feature_matrix.shape}')

In [ ]:
# Genre prevalence — fraction of users with each music genre
music_genre_cols = [c for c in genre_cols if c.startswith('music_')]
prevalence = {
    c.replace('music_', ''): spotify_df[c].mean()
    for c in music_genre_cols
}
prev_df = (
    pd.DataFrame(list(prevalence.items()), columns=['genre', 'user_fraction'])
    .sort_values('user_fraction', ascending=False)
)

fig = px.bar(
    prev_df,
    x='genre', y='user_fraction',
    title='Music Genre Prevalence Across Users',
    labels={'genre': 'Genre', 'user_fraction': 'Fraction of Users'},
    color='user_fraction',
    color_continuous_scale='Viridis'
)
fig.update_layout(xaxis_tickangle=45, height=450, yaxis_tickformat='.0%')
fig.show()

In [ ]:
# Audio feature distributions across all users
audio_df = spotify_df.select(AUDIO_COLS).to_pandas()
melted = audio_df.melt(var_name='feature', value_name='value')

fig = px.box(
    melted, x='feature', y='value',
    title='Scaled Audio Feature Distributions (all users)',
    labels={'feature': 'Feature', 'value': 'Scaled Value [0–1]'},
    color='feature',
    color_discrete_sequence=px.colors.qualitative.Safe
)
fig.update_layout(showlegend=False, height=420)
fig.show()

## 3. Hyperparameter Tuning: Find Optimal K

**Why PCA first?** The feature matrix contains many binary genre columns, which
are sparse and noisy. PCA removes low-variance dimensions and makes K-Means
converge faster and more stably (K-Means is sensitive to high dimensionality).

**Why three metrics?**
- **Silhouette score** [-1, 1]: measures how similar a point is to its own cluster vs.
  neighbouring clusters. Higher is better; values above 0.2 indicate meaningful structure.
- **Davies-Bouldin index** [0, ∞]: ratio of within-cluster scatter to between-cluster
  separation. Lower is better; penalises both loose and overlapping clusters.
- **Calinski-Harabasz index** [0, ∞]: ratio of between-cluster to within-cluster
  dispersion. Higher is better; tends to favour compact, well-separated clusters.

Using all three guards against cherry-picking a k that only looks good on one metric.

In [ ]:
# PCA — reduce to at most 20 components for faster, more stable clustering
n_components = min(20, feature_matrix.shape[1] - 1)
pca = PCA(n_components=n_components, random_state=42)
feature_matrix_pca = pca.fit_transform(feature_matrix)

print(f'Variance explained by {n_components} PCA components: {pca.explained_variance_ratio_.sum():.2%}')

k_values          = list(range(2, 11))
silhouette_scores = []
db_scores         = []   # Davies-Bouldin — lower is better
ch_scores         = []   # Calinski-Harabasz — higher is better
inertias          = []

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(feature_matrix_pca)

    sil = silhouette_score(feature_matrix_pca, labels)
    db  = davies_bouldin_score(feature_matrix_pca, labels)
    ch  = calinski_harabasz_score(feature_matrix_pca, labels)

    silhouette_scores.append(sil)
    db_scores.append(db)
    ch_scores.append(ch)
    inertias.append(kmeans.inertia_)

    print(f'k={k}: silhouette={sil:.4f}  DB={db:.4f}  CH={ch:.1f}  inertia={kmeans.inertia_:.1f}')

# Best k = highest silhouette (primary), corroborated by lowest DB and highest CH
best_k = k_values[int(np.argmax(silhouette_scores))]
print(f'\nBest k by silhouette: {best_k}')
print(f'  DB at best k:  {db_scores[best_k-2]:.4f}  (lower = better)')
print(f'  CH at best k:  {ch_scores[best_k-2]:.1f}  (higher = better)')


In [ ]:
# Four-panel tuning plot — all three clustering metrics + elbow
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Silhouette Score (↑ better)',
        'Davies-Bouldin Index (↓ better)',
        'Calinski-Harabasz Index (↑ better)',
        'Elbow Curve — Inertia (↓ better)',
    ]
)

def highlight(values, best_k, higher_is_better=True):
    best_idx = int(np.argmax(values)) if higher_is_better else int(np.argmin(values))
    return ['red' if i == best_idx else '#667eea' for i in range(len(values))]

for (row, col), (y, label, hib) in zip(
    [(1,1),(1,2),(2,1),(2,2)],
    [
        (silhouette_scores, 'Silhouette', True),
        (db_scores,         'DB Index',  False),
        (ch_scores,         'CH Index',  True),
        (inertias,          'Inertia',   False),
    ]
):
    fig.add_trace(
        go.Scatter(
            x=k_values, y=y,
            mode='lines+markers',
            marker=dict(color=highlight(y, best_k, hib), size=9),
            name=label
        ),
        row=row, col=col
    )

fig.update_xaxes(title_text='k')
fig.update_layout(
    height=560,
    title_text=f'K-Means Hyperparameter Tuning — Best k={best_k} (highlighted in red)',
    showlegend=False
)
fig.show()


## 4. K-Means Clustering with Optimal K

In [ ]:
# Fit final model on PCA-reduced user features
kmeans_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_labels = kmeans_final.fit_predict(feature_matrix_pca)

# Attach cluster labels to user dataframe
user_clusters_df = spotify_df.with_columns(
    pl.Series('cluster', cluster_labels)
).to_pandas()

cluster_counts = user_clusters_df.groupby('cluster').size().reset_index(name='user_count')
print('Users per cluster:')
print(cluster_counts.to_string(index=False))


In [ ]:
# 2D PCA scatter — one point per user, coloured by cluster
pca_2d = PCA(n_components=2, random_state=42)
coords = pca_2d.fit_transform(feature_matrix)

# Build a readable 'top music genres' string per user for hover
music_genre_names = [c.replace('music_', '') for c in genre_cols if c.startswith('music_')]
def top_genres(row, prefix, n=3):
    scores = {g: row[f'{prefix}_{g}'] for g in music_genre_names if f'{prefix}_{g}' in row}
    return ', '.join([g for g, v in sorted(scores.items(), key=lambda x: -x[1]) if v > 0][:n]) or 'none'

user_clusters_df['top_music'] = user_clusters_df.apply(lambda r: top_genres(r, 'music'), axis=1)

viz_df = pd.DataFrame({
    'pc1':       coords[:, 0],
    'pc2':       coords[:, 1],
    'cluster':   cluster_labels.astype(str),
    'user_id':   user_clusters_df['user_id'],
    'top_music': user_clusters_df['top_music'],
    'avg_energy':       user_clusters_df['avg_energy'].round(3),
    'avg_danceability': user_clusters_df['avg_danceability'].round(3),
})

fig = px.scatter(
    viz_df, x='pc1', y='pc2',
    color='cluster',
    title=f'User Clusters (k={best_k}) — PCA 2D Projection',
    labels={'pc1': 'PC 1', 'pc2': 'PC 2'},
    hover_data=['user_id', 'top_music', 'avg_energy', 'avg_danceability'],
    color_discrete_sequence=px.colors.qualitative.Bold
)
fig.update_layout(height=520)
fig.show()


In [ ]:
# Per-cluster summary: mean audio features + top genre preferences
feature_df = pd.DataFrame(feature_matrix, columns=feature_cols)
feature_df['cluster'] = cluster_labels

print(f'Top audio features and genre preferences per cluster:\n')
for c in sorted(feature_df['cluster'].unique()):
    cdata = feature_df[feature_df['cluster'] == c]

    # Mean audio features
    audio_means = cdata[AUDIO_COLS].mean().sort_values(ascending=False)
    audio_str = ', '.join([f'{col.replace("avg_","")} {v:.2f}' for col, v in audio_means.items()])

    # Top music genres by mean presence
    music_cols_here = [c2 for c2 in genre_cols if c2.startswith('music_')]
    top_music = (
        cdata[music_cols_here].mean()
        .sort_values(ascending=False)
        .head(4)
    )
    music_str = ', '.join([f'{col.replace("music_","")} ({v:.0%})' for col, v in top_music.items()])

    print(f'Cluster {c}  ({len(cdata)} users)')
    print(f'  Audio:        {audio_str}')
    print(f'  Top music:    {music_str}')
    print()


## 5. Feature Importance: Which Genres Distinguish Clusters?

A **Random Forest classifier** predicts cluster membership from the full feature
matrix. This lets us identify which features most define the cluster boundaries.

**Why Random Forest?** Tree ensembles handle mixed feature types (continuous audio
features + binary genre flags) without scaling, and Gini importances are a
well-understood, model-native importance measure.

**Hyperparameter rationale:**
| Parameter | Range searched | Why |
|---|---|---|
| `n_estimators` | 50, 100, 200 | More trees reduce variance at compute cost; 100 is a common sweet spot |
| `max_depth` | None, 5, 10, 20 | Unlimited depth risks overfitting; shallow trees act as weak learners |
| `min_samples_split` | 2, 5, 10 | Higher values regularise the tree, preventing memorisation of small clusters |
| `max_features` | sqrt, log2, 0.3 | Subsetting features per split introduces diversity and reduces correlation across trees |

**Overfitting check:** We compare training accuracy to 5-fold CV accuracy.
A large gap (> 0.10) signals overfitting and would call for stronger regularisation.

In [ ]:
# Baseline random forest — train accuracy vs CV to check for overfitting
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(feature_matrix, cluster_labels)

train_acc = rf.score(feature_matrix, cluster_labels)
cv_scores = cross_val_score(rf, feature_matrix, cluster_labels, cv=5, scoring='balanced_accuracy')

print(f'Baseline RF')
print(f'  Train accuracy:          {train_acc:.3f}')
print(f'  CV balanced accuracy:    {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')
print(f'  Overfit gap:             {train_acc - cv_scores.mean():.3f}  (> 0.10 would be concerning)')


In [ ]:
# RandomizedSearchCV — tune RF hyperparameters
# n_iter=15 samples the space broadly without exhaustive cost (grid would be 3×4×3×3=108 fits)
param_dist = {
    'n_estimators':     [50, 100, 200],       # more trees = lower variance
    'max_depth':        [None, 5, 10, 20],    # None = full depth (may overfit)
    'min_samples_split':[2, 5, 10],           # higher = stronger regularisation
    'max_features':     ['sqrt', 'log2', 0.3],# controls feature subsampling per split
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=15,
    cv=5,                          # 5-fold for stable estimates
    scoring='balanced_accuracy',   # balanced_accuracy corrects for cluster size imbalance
    random_state=42,
    verbose=1,
    return_train_score=True,       # capture train score to check overfitting
)
rf_search.fit(feature_matrix, cluster_labels)

best = rf_search.best_estimator_
tuned_train_acc = best.score(feature_matrix, cluster_labels)
tuned_cv_scores = cross_val_score(best, feature_matrix, cluster_labels, cv=5, scoring='balanced_accuracy')

print(f'\nBest params: {rf_search.best_params_}')
print(f'\nTuned RF')
print(f'  Train accuracy:          {tuned_train_acc:.3f}')
print(f'  CV balanced accuracy:    {tuned_cv_scores.mean():.3f} ± {tuned_cv_scores.std():.3f}')
print(f'  Overfit gap:             {tuned_train_acc - tuned_cv_scores.mean():.3f}')
print(f'  Improvement over baseline: +{tuned_cv_scores.mean() - cv_scores.mean():.3f}')


In [ ]:
# Full classification report — precision, recall, F1 per cluster
# (balanced_accuracy alone can hide per-class weaknesses)
best_rf = rf_search.best_estimator_
y_pred = best_rf.predict(feature_matrix)

print('Tuned RF — Classification Report (train set, for per-cluster breakdown):')
print(classification_report(cluster_labels, y_pred,
                             target_names=[f'Cluster {c}' for c in sorted(set(cluster_labels))]))

# 5-fold balanced accuracy already gives out-of-sample estimates above;
# this report shows per-cluster precision/recall to catch weak spots.


In [ ]:
# Feature importances from tuned model — top 20
best_rf = rf_search.best_estimator_

importances_df = pd.DataFrame({
    'feature':    feature_cols,
    'importance': best_rf.feature_importances_
}).sort_values('importance', ascending=False).head(20)

# Label each bar by its type
def feat_type(name):
    if name.startswith('avg_'):     return 'audio'
    if name.startswith('music_'):   return 'music genre'
    if name.startswith('podcast_'): return 'podcast genre'
    return 'book genre'

importances_df['type'] = importances_df['feature'].map(feat_type)

fig = px.bar(
    importances_df,
    x='importance', y='feature',
    orientation='h',
    color='type',
    title='Top 20 Feature Importances (Tuned Random Forest)',
    labels={'importance': 'Feature Importance (Gini)', 'feature': 'Feature'},
    color_discrete_map={
        'audio':          '#667eea',
        'music genre':    '#764ba2',
        'podcast genre':  '#f093fb',
        'book genre':     '#4facfe',
    }
)
fig.update_layout(height=560, yaxis={'categoryorder': 'total ascending'})
fig.show()


In [ ]:
# Baseline vs tuned importances side by side — top 15 by tuned importance
compare_df = pd.DataFrame({
    'feature':  feature_cols,
    'baseline': rf.feature_importances_,
    'tuned':    best_rf.feature_importances_
}).sort_values('tuned', ascending=False).head(15)

fig = go.Figure()
fig.add_trace(go.Bar(
    name='Baseline RF',
    y=compare_df['feature'], x=compare_df['baseline'],
    orientation='h', marker_color='#667eea'
))
fig.add_trace(go.Bar(
    name='Tuned RF',
    y=compare_df['feature'], x=compare_df['tuned'],
    orientation='h', marker_color='#764ba2'
))
fig.update_layout(
    barmode='group',
    title='Baseline vs Tuned Random Forest: Top Feature Importances',
    xaxis_title='Importance (Gini)',
    yaxis=dict(categoryorder='total ascending'),
    height=500
)
fig.show()


## 6. Summary

| Step | What we did | Key result |
|---|---|---|
| **Load Data** | Read `data/user_features.csv` from preprocessing | 300 users × (6 audio + n genre) features |
| **Feature Overview** | Genre prevalence bar chart + audio box plots | Identified dominant genres and feature spread |
| **K-Means Tuning** | Silhouette + Davies-Bouldin + Calinski-Harabasz sweep over k=2..10 | Three metrics cross-validated best k selection |
| **Clustering** | K-Means on PCA-reduced matrix with best k | Users grouped into coherent audio+genre clusters |
| **RF Baseline** | Train vs 5-fold CV balanced accuracy | Quantified overfitting gap before tuning |
| **RF Tuning** | RandomizedSearchCV (n_iter=15, cv=5, balanced_accuracy) | Improved CV accuracy; overfitting gap monitored |
| **Evaluation** | Per-cluster classification report (precision/recall/F1) | Identified any weak clusters, not just mean accuracy |